<a href="https://colab.research.google.com/github/faisu6339-glitch/RAG/blob/main/Vector_Databases.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Vector Databases: A Detailed Explanation

A **Vector Database** is a specialized type of database designed to store, index, and query high-dimensional vector data (often called **vector embeddings**).

Unlike traditional relational databases that store structured data in rows and columns, or document databases that store semi-structured JSON, vector databases are optimized for handling complex, unstructured data (like text, images, audio, and video) converted into numerical representations.

---

## 1. What are Vector Embeddings?
To understand vector databases, you must first understand **vector embeddings**:
* **Definition:** An embedding is a representation of unstructured data as a list of numbers (a vector) in a continuous high-dimensional space.
* **Semantic Meaning:** Deep learning models (like BERT, CLIP, or OpenAI's text-embedding-models) translate data into these vectors such that **conceptually similar items are located close to each other** in the vector space.
* **Example:** The vector for "king" will be geometrically closer to "queen" than to "apple".

---

## 2. Key Differences: Traditional vs. Vector Databases

| Feature | Traditional Databases (SQL/NoSQL) | Vector Databases |
| :--- | :--- | :--- |
| **Data Type** | Structured (tabular, text, JSON) | High-dimensional vectors (embeddings) |
| **Search Type** | Exact match (keyword search, exact values) | Similarity search (find items conceptually "close" to the query) |
| **Query Method** | SQL queries, key-value lookups | Distance metrics (Cosine, Euclidean, Dot Product) |
| **Index Type** | B-Trees, Hash indexes | ANNS (Approximate Nearest Neighbor Search) |

---

## 3. How a Vector Database Works

1. **Data Ingestion & Embedding:** Raw unstructured data (e.g., a PDF document) is passed through an embedding model to generate a high-dimensional vector.
2. **Indexing:** The vector database stores both the vector and the associated metadata. It indexes the vectors using advanced algorithms to enable ultra-fast retrieval.
3. **Querying:** When a user searches (e.g., "How do I reset my password?"), the query is converted into an embedding vector using the same model.
4. **Similarity Search:** The database compares the query vector to the stored vectors using similarity metrics and returns the closest matches.

---

## 4. Common Indexing Algorithms
Because searching billions of high-dimensional vectors sequentially (flat search) is too slow, vector databases use **Approximate Nearest Neighbor Search (ANNS)** algorithms:
* **HNSW (Hierarchical Navigable Small World):** A graph-based index that allows very fast search speeds with high accuracy.
* **IVF (Inverted File Index):** Segments the vector space into clusters to reduce search space.
* **PQ (Product Quantization):** Compresses vectors to reduce memory consumption while sacrificing a small amount of accuracy.

---

## 5. Primary Use Cases

* **Retrieval-Augmented Generation (RAG):** Providing LLMs with external, up-to-date knowledge bases to prevent hallucinations.
* **Semantic Search:** Understanding user intent behind search queries rather than just matching literal keywords.
* **Recommendation Engines:** Suggesting products, movies, or music based on user behavior embeddings.
* **Anomaly Detection:** Identifying patterns or vectors that deviate significantly from standard clusters (used in fraud detection).

---

## 6. Popular Vector Databases
* **Dedicated Vector Databases:** Milvus, Pinecone, Qdrant, Weaviate, Chroma.
* **Vector Plugins for Existing Databases:** pgvector (for PostgreSQL), Redis, Elasticsearch.

In [1]:

!pip install -q langchain-nvidia-ai-endpoints chromadb faiss-cpu numpy pandas scikit-learn


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.


In [2]:

from google.colab import userdata
import os

os.environ["NVIDIA_API_KEY"] = userdata.get("NVIDIA_API_KEY")

print("NVIDIA API key configured successfully")


NVIDIA API key configured successfully


In [5]:
import requests
from google.colab import userdata

api_key = userdata.get("NVIDIA_API_KEY")

url = "https://integrate.api.nvidia.com/v1/embeddings"

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

text = "Artificial Intelligence is transforming technology."

payload = {
    "input": text,
    "model": "nvidia/nemotron-3.5-embed-1b",
    "input_type": "query",
    "encoding_format": "float",
    "truncate": "END"
}

response = requests.post(
    url,
    headers=headers,
    json=payload
)

response.raise_for_status()

result = response.json()

vector = result["data"][0]["embedding"]

print("Embedding dimensions:", len(vector))
print("First 10 values:", vector[:10])

Embedding dimensions: 2048
First 10 values: [0.107421875, -0.0033493042, -0.02218628, 0.009063721, 0.012245178, -0.0009212494, 0.01953125, -0.005897522, 0.004699707, 0.016799927]


In [7]:
import requests
from google.colab import userdata
from langchain_core.embeddings import Embeddings

class NVIDIAEmbeddingModel(Embeddings):

    def __init__(self, api_key):
        self.api_key = api_key
        self.url = "https://integrate.api.nvidia.com/v1/embeddings"
        self.model = "nvidia/nemotron-3.5-embed-1b"

    def _embed(self, texts, input_type):

        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }

        payload = {
            "input": texts,
            "model": self.model,
            "input_type": input_type,
            "encoding_format": "float",
            "truncate": "END"
        }

        response = requests.post(
            self.url,
            headers=headers,
            json=payload
        )

        response.raise_for_status()

        data = response.json()["data"]

        return [
            item["embedding"]
            for item in sorted(data, key=lambda x: x["index"])
        ]

    def embed_documents(self, texts):
        return self._embed(texts, "passage")

    def embed_query(self, text):
        return self._embed([text], "query")[0]


embedding_model = NVIDIAEmbeddingModel(
    api_key=userdata.get("NVIDIA_API_KEY")
)

# Program 1: Store documents as vectors using ChromaDB

In [9]:
import chromadb

client = chromadb.Client()

collection = client.get_or_create_collection(
    name="rag_documents"
)

print("Collection ready!")

documents = [
    "Python is a programming language.",
    "Machine learning helps computers learn from data.",
    "Deep learning uses neural networks.",
    "SQL is used to manage relational databases.",
    "Natural language processing works with human language."
]

# Generate embeddings
embeddings = embedding_model.embed_documents(documents)

# Store documents and vectors
collection.add(
    ids=["doc1", "doc2", "doc3", "doc4", "doc5"],
    documents=documents,
    embeddings=embeddings
)

print("Documents stored successfully!")
print("Total documents:", collection.count())

Collection ready!
Documents stored successfully!
Total documents: 5


In [10]:
results = collection.get(
    include=["documents", "embeddings"]
)

print("Total documents:", len(results["ids"]))

for doc_id, document in zip(results["ids"], results["documents"]):
    print(f"{doc_id}: {document}")

Total documents: 5
doc1: Python is a programming language.
doc2: Machine learning helps computers learn from data.
doc3: Deep learning uses neural networks.
doc4: SQL is used to manage relational databases.
doc5: Natural language processing works with human language.


In [11]:

import sqlite3

# Traditional database
connection = sqlite3.connect(":memory:")
cursor = connection.cursor()

cursor.execute("""
CREATE TABLE documents (
    id INTEGER PRIMARY KEY,
    content TEXT
)
""")

documents = [
    "Python is a programming language.",
    "Machine learning helps computers learn from data.",
    "Deep learning uses neural networks.",
    "SQL manages relational databases."
]

for document in documents:
    cursor.execute(
        "INSERT INTO documents (content) VALUES (?)",
        (document,)
    )

connection.commit()

# Exact keyword search
cursor.execute("""
SELECT content
FROM documents
WHERE content LIKE '%machine learning%'
""")

print("Traditional database results:")
for result in cursor.fetchall():
    print(result[0])

# Vector database search
query = "How do computers learn from examples?"

query_vector = embedding_model.embed_query(query)

results = collection.query(
    query_embeddings=[query_vector],
    n_results=3
)

print("\nVector database results:")
for document in results["documents"][0]:
    print(document)


Traditional database results:
Machine learning helps computers learn from data.

Vector database results:
Machine learning helps computers learn from data.
Deep learning uses neural networks.
Natural language processing works with human language.


# 3. Vector Indexing

# Program 3: Create a vector collection and index

In [12]:

import chromadb

client = chromadb.Client()

collection = client.create_collection(
    name="indexed_documents",
    metadata={"hnsw:space": "cosine"}
)

documents = [
    "Python programming",
    "Java programming",
    "Machine learning",
    "Deep learning",
    "Database management",
    "Computer networks",
    "Operating systems"
]

vectors = embedding_model.embed_documents(documents)

collection.add(
    ids=[f"id{i}" for i in range(len(documents))],
    documents=documents,
    embeddings=vectors
)

print("Vector collection created")
print("Indexed records:", collection.count())


Vector collection created
Indexed records: 7


# Program 4: Search for similar documents

In [13]:

query = "I want to learn artificial intelligence"

query_vector = embedding_model.embed_query(query)

results = collection.query(
    query_embeddings=[query_vector],
    n_results=3
)

for document in results["documents"][0]:
    print(document)


Machine learning
Deep learning
Java programming


# Program 5: Display similarity search results with distances

In [14]:

query = "How can I build neural networks?"

query_vector = embedding_model.embed_query(query)

results = collection.query(
    query_embeddings=[query_vector],
    n_results=3,
    include=["documents", "distances"]
)

for document, distance in zip(
    results["documents"][0],
    results["distances"][0]
):
    print("Document:", document)
    print("Distance:", distance)
    print("-" * 50)


Document: Deep learning
Distance: 0.7361199259757996
--------------------------------------------------
Document: Computer networks
Distance: 0.7364843487739563
--------------------------------------------------
Document: Machine learning
Distance: 0.7621142864227295
--------------------------------------------------


# Program 6: Retrieve the top 2 documents

In [15]:

query = "Explain machine learning"

query_vector = embedding_model.embed_query(query)

k = 2

results = collection.query(
    query_embeddings=[query_vector],
    n_results=k
)

print(f"Top {k} relevant documents:\n")

for rank, document in enumerate(
    results["documents"][0], start=1
):
    print(f"{rank}. {document}")


Top 2 relevant documents:

1. Machine learning
2. Deep learning


# Program 7: Make a reusable Top-K search function

In [16]:

def search_top_k(query, k=3):

    query_vector = embedding_model.embed_query(query)

    results = collection.query(
        query_embeddings=[query_vector],
        n_results=k,
        include=["documents", "distances"]
    )

    for rank, (document, distance) in enumerate(
        zip(
            results["documents"][0],
            results["distances"][0]
        ),
        start=1
    ):
        print(f"Rank: {rank}")
        print(f"Document: {document}")
        print(f"Distance: {distance:.4f}")
        print("-" * 40)


search_top_k(
    "What is deep learning?",
    k=3
)


Rank: 1
Document: Deep learning
Distance: 0.2995
----------------------------------------
Rank: 2
Document: Machine learning
Distance: 0.6147
----------------------------------------
Rank: 3
Document: Database management
Distance: 0.8137
----------------------------------------


# Program 8: Store documents with metadata

In [17]:

import chromadb

client = chromadb.Client()

collection = client.create_collection(
    name="metadata_documents"
)

documents = [
    "Python is used in machine learning.",
    "SQL is used for database management.",
    "Neural networks are used in deep learning.",
    "HTML is used to structure web pages.",
    "Computer vision analyzes images."
]

metadatas = [
    {"category": "AI", "year": 2025},
    {"category": "Database", "year": 2024},
    {"category": "AI", "year": 2025},
    {"category": "Web", "year": 2023},
    {"category": "AI", "year": 2024}
]

vectors = embedding_model.embed_documents(documents)

collection.add(
    ids=["d1", "d2", "d3", "d4", "d5"],
    documents=documents,
    embeddings=vectors,
    metadatas=metadatas
)

print("Documents and metadata stored!")


Documents and metadata stored!


# Program 9: Filter documents by category

In [18]:

results = collection.get(
    where={"category": "AI"},
    include=["documents", "metadatas"]
)

for document, metadata in zip(
    results["documents"],
    results["metadatas"]
):
    print(document)
    print(metadata)
    print("-" * 30)


Python is used in machine learning.
{'category': 'AI', 'year': 2025}
------------------------------
Neural networks are used in deep learning.
{'year': 2025, 'category': 'AI'}
------------------------------
Computer vision analyzes images.
{'category': 'AI', 'year': 2024}
------------------------------


# Program 10: Filter documents by year

In [19]:

results = collection.get(
    where={"year": {"$gte": 2025}},
    include=["documents", "metadatas"]
)

for document, metadata in zip(
    results["documents"],
    results["metadatas"]
):
    print(document, metadata)


Python is used in machine learning. {'category': 'AI', 'year': 2025}
Neural networks are used in deep learning. {'year': 2025, 'category': 'AI'}


# Program 11: Combine metadata filtering with similarity search

In [20]:

query = "Explain artificial intelligence"

query_vector = embedding_model.embed_query(query)

results = collection.query(
    query_embeddings=[query_vector],
    n_results=3,
    where={"category": "AI"},
    include=["documents", "metadatas", "distances"]
)

for document, metadata, distance in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
):
    print("Document:", document)
    print("Metadata:", metadata)
    print("Distance:", distance)
    print("-" * 40)


Document: Computer vision analyzes images.
Metadata: {'year': 2024, 'category': 'AI'}
Distance: 1.455182671546936
----------------------------------------
Document: Neural networks are used in deep learning.
Metadata: {'year': 2025, 'category': 'AI'}
Distance: 1.5220638513565063
----------------------------------------
Document: Python is used in machine learning.
Metadata: {'category': 'AI', 'year': 2025}
Distance: 1.600740671157837
----------------------------------------
